In [ ]:
# ==============================================================================
# PARALLEL DIM: STAFF
# ==============================================================================
from notebooks.helpers import IncrementalPipeline, TableConfig, get_latest_batch_id, setup_logger, write_gold_table, safe_count, generate_batch_id
import pandas as pd
from notebooks.helpers.silver_transforms import transform_staff_full_pipeline, build_staff_hierarchy_bridge

logger = setup_logger("parallel_dim_staff")

batch_id = generate_batch_id()
pipeline = IncrementalPipeline(spark, dbutils, batch_id=batch_id)

dependencies = ["address", "city", "country"]

bronze_batch_id = get_latest_batch_id(spark, "staff")
if not bronze_batch_id:
    raise ValueError("No bronze batch_id found for staff; run bronze load first.")
logger.info(f"Using bronze batch_id for staff: {bronze_batch_id}")

config = TableConfig(
    table_name="staff",
    business_key="staff_id",
    surrogate_key="staff_key",
    watermark_column="last_update",
    scd_type=1,
    gold_table_name="dim_staff",
    silver_transform=transform_staff_full_pipeline,
    dependencies=dependencies,
)

print("Row Counts (Before):")
print(f"dim_staff: {safe_count(spark, 'dim_staff')}")
print(f"bridge_staff_hierarchy: {safe_count(spark, 'bridge_staff_hierarchy')}")
print("\nLatest Watermarks (Before):")
display(spark.table("wheelie.monitoring.watermarks"))

results = pipeline.load_tables([config], force_full=False, bronze_batch_id=bronze_batch_id)

# Rebuild staff hierarchy bridge after staff load
staff_bronze = spark.table("wheelie.bronze.staff")
bridge_staff_hierarchy = build_staff_hierarchy_bridge(staff_bronze, max_depth=10)
write_gold_table(bridge_staff_hierarchy, "bridge_staff_hierarchy", mode="overwrite")

logger.info("bridge_staff_hierarchy rebuilt")
display(pd.DataFrame(results))

print("\nRow Counts (After):")
print(f"dim_staff: {safe_count(spark, 'dim_staff')}")
print(f"bridge_staff_hierarchy: {safe_count(spark, 'bridge_staff_hierarchy')}")
print("\nLatest Watermarks (After):")
display(spark.table("wheelie.monitoring.watermarks"))
